In [2]:
!pip3 install deltalake

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.7/44.7 MB 10.6 MB/s eta 0:00:000:00:010:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 10.8 MB/s eta 0:00:0031m11.7 MB/s eta 0:00:01

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: pip3 install --upgrade pip


In [3]:
from deltalake import DeltaTable
from deltalake.writer import write_deltalake
import pandas as pd

In [ ]:
def merge_cdc_changes(target_uri, new_changes_df):
    dt = DeltaTable(target_uri)

    (
        dt.merge(
            source=new_changes_df,
            predicate="target.user_id = source.user_id",
            source_alias="source",
            target_alias="target"
        )
        .when_matched_update(updates={
            "email": "source.email",
            "updated_at": "source.updated_at"
        })
        .when_not_matched_insert(updates={
            "user_id": "source.user_id",
            "email": "source.email",
            "created_at": "source.updated_at",
            "updated_at": "source.updated_at"
        })
        .execute()
    )